In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split 

import numpy as np
import scipy.stats as stats 

import pandas as pd

import matplotlib.pyplot as plt

In [26]:
data = pd.read_csv('data/winequality-white.csv', sep=";")
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [27]:
cols2zscore = data.keys()
cols2zscore = cols2zscore.drop('quality')
data[cols2zscore] = data[cols2zscore].apply(stats.zscore)

data['boolQuality'] = 0
data.loc[ data['quality'] > 5, 'boolQuality' ] = 1

In [30]:
dataT = torch.tensor(data[cols2zscore].values).float()
labels = torch.tensor(data['boolQuality'].values).float()
labels = labels[:,None]

train_data, test_data, train_labels, test_labels = train_test_split(dataT, labels, test_size=0.2)

train_dataDataset = TensorDataset(train_data, train_labels)
test_dataDataset = TensorDataset(test_data, test_labels)

train_loader = DataLoader(train_dataDataset, batch_size=32, shuffle=32, drop_last=True)
test_loader = DataLoader(test_dataDataset, batch_size=32, drop_last=True)

In [35]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,boolQuality
0,0.172097,-0.081770,0.213280,2.821349,-0.035355,0.569932,0.744565,2.331512,-1.246921,-0.349184,-1.393152,6,1
1,-0.657501,0.215896,0.048001,-0.944765,0.147747,-1.253019,-0.149685,-0.009154,0.740029,0.001342,-0.824276,6,1
2,1.475751,0.017452,0.543838,0.100282,0.193523,-0.312141,-0.973336,0.358665,0.475102,-0.436816,-0.336667,6,1
3,0.409125,-0.478657,-0.117278,0.415768,0.559727,0.687541,1.121091,0.525855,0.011480,-0.787342,-0.499203,6,1
4,0.409125,-0.478657,-0.117278,0.415768,0.559727,0.687541,1.121091,0.525855,0.011480,-0.787342,-0.499203,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,-0.776015,-0.677101,-0.365197,-0.944765,-0.310008,-0.664970,-1.091000,-0.965483,0.541334,0.088973,0.557282,6,1
4894,-0.301959,0.414339,0.213280,0.317179,0.056196,1.275590,0.697499,0.291789,-0.253446,-0.261553,-0.743008,5,0
4895,-0.420473,-0.379435,-1.191592,-1.023637,-0.218457,-0.312141,-0.643875,-0.497350,-1.313153,-0.261553,-0.905544,6,1
4896,-1.605613,0.116674,-0.282557,-1.043355,-1.088192,-0.900190,-0.667408,-1.784717,1.004955,-0.962605,1.857572,7,1


In [36]:
class ANNwine(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.input = nn.Linear(11,16)
        
        self.fc1 = nn.Linear(16,32)
        self.fc2 = nn.Linear(32,32)
        
        self.output = nn.Linear(32,1)
    
    def forward(self, x):
        x = F.relu( self.input(x) )
        x = F.relu( self.fc1(x) )
        x = F.relu( self.fc2(x) )
        return self.output(x)

In [39]:
numepochs = 1000


def trainTheModel():
    
    model = ANNwine().to('cuda')
    lossfun = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    
    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []
    
    # Iterate Epochs
    for epochi in range(numepochs):
        
        # === Train === #
        model.train()
        
        batchAcc = []
        batchLoss = []
        
        for X, y in train_loader:
            X, y = X.to('cuda'), y.to('cuda')
            
            yHat = model(X)
            loss = lossfun(yHat, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            batchLoss.append(loss.item())
            
            batchAcc.append( 100*torch.mean(((yHat>0) == y).float()).item() )
            
        trainAcc.append( np.mean(batchAcc) )
        
        losses[epochi] = np.mean(batchLoss)
        
        X, y = next(iter(test_loader))
        X, y = X.to('cuda'), y.to('cuda')
        with torch.no_grad():
            yHat = model(X)
        testAcc.append( 100*torch.mean(((yHat>0) == y).float()).item() )
        
    return model, trainAcc, testAcc, losses

In [40]:
model, trainAcc, testAcc, losses = trainTheModel()

In [43]:
train_predictions = model(train_loader.dataset.tensors[0].to('cuda'))
train_predictions

# X, y = iter(next(test_loader)) v v v
test_predictions = model(test_loader.dataset.tensors[0].to('cuda'))
test_predictions

tensor([[ 2.0574e+00],
        [-2.4601e+00],
        [ 2.8189e+00],
        [ 1.1623e+01],
        [ 2.7740e+00],
        [-2.6644e-01],
        [-2.0760e+00],
        [ 1.2915e+00],
        [ 2.8043e+00],
        [-1.6293e+00],
        [ 6.0929e+00],
        [ 3.0730e+00],
        [ 1.8342e+00],
        [ 3.4208e+00],
        [-1.5801e+01],
        [ 5.1570e+00],
        [ 2.4840e+00],
        [ 8.9628e+00],
        [ 2.4646e+00],
        [ 3.5685e+00],
        [ 5.5409e+00],
        [-1.8303e+00],
        [-3.3293e+00],
        [ 3.3545e+00],
        [ 1.7153e+00],
        [ 1.0291e+01],
        [-1.8228e+00],
        [ 6.0074e+00],
        [-1.0674e+01],
        [-1.6905e+00],
        [ 9.6339e-01],
        [ 1.6360e+00],
        [ 9.6261e+00],
        [-6.5144e-01],
        [-1.6388e+00],
        [ 5.0452e+00],
        [ 7.1243e-01],
        [ 1.5888e+00],
        [ 2.9781e+00],
        [ 3.6020e+00],
        [ 6.2804e+00],
        [ 3.1971e+00],
        [ 4.8375e+00],
        [ 9

In [49]:
import sklearn.metrics as skm

train_metrics = [0,0,0,0]
test_metrics = [0,0,0,0]

train_metrics[0] = skm.accuracy_score(train_loader.dataset.tensors[1].cpu(), train_predictions.cpu()>0)
train_metrics[1] = skm.precision_score(train_loader.dataset.tensors[1].cpu(), train_predictions.cpu()>0)
train_metrics[2] = skm.recall_score(train_loader.dataset.tensors[1].cpu(), train_predictions.cpu()>0)
train_metrics[3] = skm.f1_score(train_loader.dataset.tensors[1].cpu(), train_predictions.cpu()>0)

test_metrics[0] = skm.accuracy_score(test_loader.dataset.tensors[1].cpu(), test_predictions.cpu() > 0)
test_metrics[1] = skm.precision_score(test_loader.dataset.tensors[1].cpu(), test_predictions.cpu() > 0)
test_metrics[2] = skm.recall_score(test_loader.dataset.tensors[1].cpu(), test_predictions.cpu() > 0)
test_metrics[3] = skm.f1_score(test_loader.dataset.tensors[1].cpu(), test_predictions.cpu() > 0)

In [51]:
train_metrics

[0.8846350178662583,
 0.9017622797150356,
 0.9267822736030829,
 0.9141011022424933]

In [52]:
test_metrics

[0.7663265306122449,
 0.8248502994011976,
 0.8310708898944194,
 0.8279489105935387]